In [ ]:
import time
import json
import boto3
import base64

from pathlib import Path
from PIL import Image
from IPython.display import display

MODEL_ID = "us.anthropic.claude-sonnet-4-6"
MAX_TOKENS = 1000

bedrock_client = boto3.client('bedrock-runtime')
rekognition_client = boto3.client('rekognition')

In [ ]:
def _media_type_for(path: str) -> str:
    ext = Path(path).suffix.lower().lstrip(".")
    return {"jpg": "image/jpeg", "jpeg": "image/jpeg", "png": "image/png",
            "gif": "image/gif", "webp": "image/webp"}.get(ext, "image/jpeg")


def encode_image(label_image_path: str):
    with open(label_image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


def get_rekognition_labels(label_image_path):
    with open(label_image_path, "rb") as image_file:
        image_bytes = image_file.read()

    response = rekognition_client.detect_labels(
        Image={"Bytes": image_bytes},
        MaxLabels=10,
        MinConfidence=75,
    )
    return response["Labels"]


def claude_response(base64_img, prompt, media_type="image/jpeg"):
    body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": MAX_TOKENS,
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "image",
                     "source": {"type": "base64", "media_type": media_type, "data": base64_img}},
                    {"type": "text", "text": prompt},
                ],
            }
        ],
    }

    response = bedrock_client.invoke_model(
        modelId=MODEL_ID,
        contentType="application/json",
        accept="application/json",
        body=json.dumps(body),
    )

    return json.loads(response["body"].read())

In [ ]:
label_image_path = r'images/label_detection/skateboard_resized.jpg'
display(Image.open(label_image_path))

In [ ]:
start_time = time.time()
labels = get_rekognition_labels(label_image_path)
rekognition_time = time.time() - start_time
print('Rekognition Response: \n')
for label in labels:
    print(f"Label: {label['Name']}, Confidence: {label['Confidence']:.2f}%")
print(f"\nTime taken for Rekognition: {rekognition_time:.2f} seconds")


base64_img = encode_image(label_image_path)
prompt = "What's in this image, only give the labels"

start_time = time.time()
claud = claude_response(base64_img, prompt, _media_type_for(label_image_path))
claude_time = time.time() - start_time

print('\nClaude  Response: \n')
print(claud['content'][0]['text'])
print(f"\nTime taken for Claude: {claude_time:.2f} seconds")

In [ ]:
def get_rekognition_image_properties(label_image_path):
    with open(label_image_path, 'rb') as image_file:
        image_bytes = image_file.read()

    response = rekognition_client.detect_labels(
        Image={'Bytes': image_bytes},
        Features=[
        'IMAGE_PROPERTIES'],
        Settings = {
       'ImageProperties': {
                'MaxDominantColors': 5
            } 
        }
    )
    return response['ImageProperties']



def print_rekognition_image_properties(data):
    print("Quality:")
    for key, value in data["Quality"].items():
        print(f"  {key}: {value:.2f}")

    print("\nDominant Colors:")
    for color in data["DominantColors"]:
        print(f"  Color: {color['CSSColor']} (Hex: {color['HexCode']}, RGB: ({color['Red']}, {color['Green']}, {color['Blue']}), Pixel Percent: {color['PixelPercent']:.2f}%)")

    print("\nForeground Quality:")
    for key, value in data["Foreground"]["Quality"].items():
        print(f"  {key}: {value:.2f}")
    print("\nForeground Dominant Colors:")
    for color in data["Foreground"]["DominantColors"]:
        print(f"  Color: {color['CSSColor']} (Hex: {color['HexCode']}, RGB: ({color['Red']}, {color['Green']}, {color['Blue']}), Pixel Percent: {color['PixelPercent']:.2f}%)")

    print("\nBackground Quality:")
    for key, value in data["Background"]["Quality"].items():
        print(f"  {key}: {value:.2f}")
    print("\nBackground Dominant Colors:")
    for color in data["Background"]["DominantColors"]:
        print(f"  Color: {color['CSSColor']} (Hex: {color['HexCode']}, RGB: ({color['Red']}, {color['Green']}, {color['Blue']}), Pixel Percent: {color['PixelPercent']:.2f}%)")

    return None

In [ ]:
img_properties = r'images/image_properties/green_car.jpeg'
display(Image.open(img_properties))

In [ ]:
start_time = time.time()
properties = get_rekognition_image_properties (img_properties)
rekognition_properties_time = time.time() - start_time
print_rekognition_image_properties(properties)
print(f"\nTime taken for Rekognition: {rekognition_properties_time:.2f} seconds")


base64_img = encode_image(img_properties)
prompt = '''
Give information about image quality (sharpness, brightness, and contrast) for the entire image. 
Also give sharpness and brightness are for the foreground and background of the image  
Additionally also tell the dominant colors of the entire image, foreground, background.
Along with textual descriptions give hex values of the colors with a break down of the RGB values and the percentage of 
the amount of pixels they occupy in the image.

'''
start_time = time.time()
claud = claude_response(base64_img, prompt, _media_type_for(img_properties))
claude_time = time.time() - start_time

print('\nClaude  Response: \n')
print(claud['content'][0]['text'])
print(f"\nTime taken for Claude: {claude_time:.2f} seconds")

In [ ]:
def get_rekognition_faces(label_image_path):
    with open(label_image_path, 'rb') as image_file:
        image_bytes = image_file.read()

    response = rekognition_client.detect_faces(
        Image={'Bytes': image_bytes},
        Attributes=[
        'DEFAULT','ALL','AGE_RANGE','BEARD','EMOTIONS','EYE_DIRECTION','EYEGLASSES','EYES_OPEN','GENDER','MOUTH_OPEN','MUSTACHE','FACE_OCCLUDED','SMILE','SUNGLASSES',
    ]
    )
    return response

def print_rekognition_face_details(data):
    for face in data["FaceDetails"]:
        print("Bounding Box:")
        for key, value in face["BoundingBox"].items():
            print(f"  {key}: {value:.2f}")

        print("\nAge Range:")
        print(f"  Low: {face['AgeRange']['Low']}, High: {face['AgeRange']['High']}")
        print("\nSmile:")
        print(f"  Value: {face['Smile']['Value']}, Confidence: {face['Smile']['Confidence']:.2f}")
        print("\nEyeglasses:")
        print(f"  Value: {face['Eyeglasses']['Value']}, Confidence: {face['Eyeglasses']['Confidence']:.2f}")
        print("\nSunglasses:")
        print(f"  Value: {face['Sunglasses']['Value']}, Confidence: {face['Sunglasses']['Confidence']:.2f}")
        print("\nGender:")
        print(f"  Value: {face['Gender']['Value']}, Confidence: {face['Gender']['Confidence']:.2f}")
        print("\nBeard:")
        print(f"  Value: {face['Beard']['Value']}, Confidence: {face['Beard']['Confidence']:.2f}")
        print("\nMustache:")
        print(f"  Value: {face['Mustache']['Value']}, Confidence: {face['Mustache']['Confidence']:.2f}")
        print("\nEyes Open:")
        print(f"  Value: {face['EyesOpen']['Value']}, Confidence: {face['EyesOpen']['Confidence']:.2f}")
        print("\nMouth Open:")
        print(f"  Value: {face['MouthOpen']['Value']}, Confidence: {face['MouthOpen']['Confidence']:.2f}")

        print("\nEmotions:")
        for emotion in face["Emotions"]:
            print(f"  Type: {emotion['Type']}, Confidence: {emotion['Confidence']:.2f}")

        # print("\nLandmarks:")
        # for landmark in face["Landmarks"]:
        #     print(f"  {landmark['Type']}: (X: {landmark['X']}, Y: {landmark['Y']})")

        print("\nPose:")
        print(f"  Roll: {face['Pose']['Roll']:.2f}, Yaw: {face['Pose']['Yaw']:.2f}, Pitch: {face['Pose']['Pitch']:.2f}")

        print("\nQuality:")
        print(f"  Brightness: {face['Quality']['Brightness']:.2f}, Sharpness: {face['Quality']['Sharpness']:.2f}")

        print("\nConfidence:")
        print(f"  {face['Confidence']}")

        print("\nFace Occluded:")
        print(f"  Value: {face['FaceOccluded']['Value']:.2f}, Confidence: {face['FaceOccluded']['Confidence']:.2f}")

        print("\nEye Direction:")
        print(f"  Yaw: {face['EyeDirection']['Yaw']:.2f}, Pitch: {face['EyeDirection']['Pitch']:.2f}, Confidence: {face['EyeDirection']['Confidence']:.2f}")

        return None


In [ ]:
face_img = r'images/face/drive_resized.jpg'
display(Image.open(face_img))

In [ ]:
start_time = time.time()
face_details = get_rekognition_faces(face_img)
rekognition_face_time = time.time() - start_time
print_rekognition_face_details(face_details)
print(f"\nTime taken for Rekognition: {rekognition_face_time:.2f} seconds")


base64_img = encode_image(face_img)
# prompt = '''
# From this image, tell me about the following:
# Age range of the person
# Gender
# Facial expressions (e.g smile)
# Is the person wearing eye glasses / sunglasses
# What sort of facial hair does the person have (e.g beard, mustache )
# Are the eyes / mouth open or closed
# Detect emotions (e.g happiness, sadness)
# The pose of the person
# Is the face of the perosn occluded 
# Also tell us about any other facial features/atributes you find
# Your response should be in the form of key value pairs

# '''



prompt = '''
tell me about the mood and sentiment of the person in the image, do a facial analyis

'''

start_time = time.time()
claud = claude_response(base64_img, prompt, _media_type_for(face_img))
claude_time = time.time() - start_time

print('\nClaude  Response: \n')
print(claud['content'][0]['text'])
print(f"\nTime taken for Claude: {claude_time:.2f} seconds")


In [ ]:
def get_rekognition_celebs(label_image_path):
    with open(label_image_path, 'rb') as image_file:
        image_bytes = image_file.read()

    response = rekognition_client.recognize_celebrities(
        Image={'Bytes': image_bytes}
    )
    return response 


def print_rekognition_celeb_details (celeb_detais):
    for celeb in celeb_detais['CelebrityFaces']:
        print(f"Name: {celeb['Name']}")
        print(f"URL: {', '.join(celeb['Urls'])}")
        print(f"ID: {celeb['Id']}")
        print(f"Match Confidence: {celeb['MatchConfidence']:.2f}%")
        print(f"Known Gender: {celeb['KnownGender']['Type']}")
        
        face = celeb['Face']
        print(f"Face Bounding Box (Left, Top, Width, Height): {face['BoundingBox']['Left']:.2f}, {face['BoundingBox']['Top']:.2f}, {face['BoundingBox']['Width']:.2f}, {face['BoundingBox']['Height']:.2f}")
        
        pose = face['Pose']
        print(f"Pose (Roll, Yaw, Pitch): {pose['Roll']:.2f}, {pose['Yaw']:.2f}, {pose['Pitch']:.2f}")
        
        quality = face['Quality']
        print(f"Face Quality (Brightness, Sharpness): {quality['Brightness']:.2f}, {quality['Sharpness']:.2f}")
        
        print("Emotions:")
        for emotion in face['Emotions']:
            print(f"  - {emotion['Type']}: {emotion['Confidence']:.2f}%")
        
        smile = face['Smile']
        print(f"Smile: {'Yes' if smile['Value'] else 'No'}, Confidence: {smile['Confidence']:.2f}%")



In [ ]:
celeb_path = r'images/celebs/30stm-top.jpg'
display(Image.open(celeb_path))

In [ ]:
start_time = time.time()
celeb_detais = get_rekognition_celebs(celeb_path)
rekognition_celeb_time = time.time() - start_time
print(f"\nTime taken for Rekognition: {rekognition_celeb_time:.2f} seconds")

print_rekognition_celeb_details(celeb_detais)


base64_img = encode_image(celeb_path)
prompt = '''
what celebrities do you see in this image ?

'''

start_time = time.time()
claud = claude_response(base64_img, prompt, _media_type_for(celeb_path))
claude_time = time.time() - start_time

print('\nClaude  Response: \n')
print(claud['content'][0]['text'])
print(f"\nTime taken for Claude: {claude_time:.2f} seconds")

In [ ]:
image_with_text = r'images/text_in_image/coffee_monday_resized.jpg'
display(Image.open(image_with_text))

In [ ]:
def get_rekognition_image_text(label_image_path):
    with open(label_image_path, 'rb') as image_file:
        image_bytes = image_file.read()

    response = rekognition_client.detect_text(
        Image={'Bytes': image_bytes}
    )
    return response


def print_rekognition_image_text (image_text_rekognition):

    for text_detection in image_text_rekognition['TextDetections']:
        print(f"Detected Text: {text_detection['DetectedText']}")
        print(f"Type: {text_detection['Type']}")
        print(f"Id: {text_detection['Id']}")
        print(f"Confidence: {text_detection['Confidence']:.2f}%")
        
        geometry = text_detection['Geometry']
        print(f"Bounding Box (Left, Top, Width, Height): {geometry['BoundingBox']['Left']:.2f}, {geometry['BoundingBox']['Top']:.2f}, {geometry['BoundingBox']['Width']:.2f}, {geometry['BoundingBox']['Height']:.2f}")
        
        print("Polygon Coordinates:")
        for point in geometry['Polygon']:
            print(f"  - ({point['X']:.2f}, {point['Y']:.2f})")
        


In [ ]:
start_time = time.time()
image_text_rekognition = get_rekognition_image_text(image_with_text)
rekognition_time = time.time() - start_time

print_rekognition_image_text(image_text_rekognition)
print(f"\nTime taken for Rekognition: {rekognition_time:.2f} seconds")

base64_img = encode_image(image_with_text)
prompt = '''
whats the text in this image and how confident are you with your detection ?
also give the bounding box coordinates for the detected text 

'''

start_time = time.time()
claud = claude_response(base64_img, prompt, _media_type_for(image_with_text))
claude_time = time.time() - start_time

print('\nClaude  Response: \n')
print(claud['content'][0]['text'])
print(f"\nTime taken for Claude: {claude_time:.2f} seconds")

In [ ]:
people_with_ppe = r'images/ppe/ppe_group_updated.jpg'
display(Image.open(people_with_ppe))

In [ ]:
def get_rekognition_ppe (label_image_path):
    with open(label_image_path, 'rb') as image_file:
        image_bytes = image_file.read()

    response = rekognition_client.detect_protective_equipment(
        Image={'Bytes': image_bytes},
        SummarizationAttributes={
        'MinConfidence': 80,
        'RequiredEquipmentTypes': [
            'FACE_COVER','HAND_COVER','HEAD_COVER'
        ]
        }
    )
    return response

def print_ppe_rekognition (ppe_rekognition):

    for person in ppe_rekognition['Persons']:
        print(f"Person Id: {person['Id']}")
        print(f"Confidence: {person['Confidence']:.2f}%")
        
        for body_part in person['BodyParts']:
            print(f"  Body Part: {body_part['Name']}")
            print(f"  Body Part Confidence: {body_part['Confidence']:.2f}%")
            
            for equipment in body_part['EquipmentDetections']:
                print(f"    Equipment Type: {equipment['Type']}")
                print(f"    Equipment Confidence: {equipment['Confidence']:.2f}%")
                
                bb = equipment['BoundingBox']
                print(f"    Bounding Box (Left, Top, Width, Height): "
                    f"{bb['Left']:.2f}, {bb['Top']:.2f}, {bb['Width']:.2f}, {bb['Height']:.2f}")
            print("-" * 40)


In [ ]:
start_time = time.time()
detect_ppe_rekognition = get_rekognition_ppe(people_with_ppe)
rekognition_time = time.time() - start_time

print_ppe_rekognition(detect_ppe_rekognition)
print (detect_ppe_rekognition)
print(f"\nTime taken for Rekognition: {rekognition_time:.2f} seconds")

base64_img = encode_image(people_with_ppe)
prompt = '''
You need to see if people in the image are wearing Personal Protective Equipment (PPE) or not.
For each person detected in the image, have a look at their face, hands and head, tell if they are covered or not.
Also tell how confident you are with your detection.

'''

start_time = time.time()
claud = claude_response(base64_img, prompt, _media_type_for(people_with_ppe))
claude_time = time.time() - start_time

print('\nClaude  Response: \n')
print(claud['content'][0]['text'])
print(f"\nTime taken for Claude: {claude_time:.2f} seconds")